# BSM L07G — Pochodzenie aplikacji, zaufanie w runtime, backup i migracja

Pracuj na projekcie `student/apps/lesson_g_app` w Android Studio.

Zaliczanie:
- `G01`: odpowiedz wysyła aplikacja automatycznie.
- `G02-G04`: wypełniasz formularz w notebooku i uruchamiasz komórkę wysyłki.


In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
# Komórka pomocnicza: formatowanie i wysyłanie odpowiedzi
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("
", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    # Wysyłka do backendu zadania
    wyslij_odpowiedz(task_id, final_answer)


# G01 — Manifest i audyt prywatności (auto-submit z aplikacji)

## Cel
Nauczyć się odróżniać:
- deklaracje w `AndroidManifest.xml` (co aplikacja *może* robić),
- realne użycie w kodzie (co aplikacja *robi*),
- prośby runtime (kiedy użytkownik widzi prompt),
- minimalizację (czy da się z mniejszym zakresem uprawnień).

## Część teoretyczna
- Uprawnienie w manifeście nie oznacza automatycznie, że aplikacja dostanie dostęp.
- Dla wielu uprawnień Android wymaga osobnej zgody użytkownika w runtime.
- Dobre praktyki:
  1. prosić o uprawnienie dopiero gdy funkcja jest potrzebna,
  2. uzasadniać w UI, po co to jest,
  3. mieć fallback (aplikacja nie powinna się „wywracać” po odmowie).

## Co masz zrobić (krok po kroku)
1. Otwórz `student/apps/lesson_g_app` w Android Studio.
1. Otwórz plik: `app/src/main/AndroidManifest.xml`.
1. Przejrzyj wszystkie wpisy `<uses-permission ...>`.
1. Otwórz plik: `app/src/main/java/com/example/secretlab/MainActivity.kt`.
1. Znajdź w UI sekcję „Student / Task 1” i zobacz, jakie uprawnienia aplikacja żąda po kliknięciu „Request permissions”.
1. Zrób mapę: każde uprawnienie -> jaka funkcja je wykorzystuje (mapa, zdjęcia, kamera, internet).
1. Sprawdź zachowanie fallback:
- co aplikacja pokazuje, gdy nie ma lokalizacji,
- co pokazuje, gdy brak dostępu do zdjęć,
- co pokazuje, gdy nie ma kamery.

## Jak zaliczasz (auto-submit)
1. Uruchom aplikację.
1. Wpisz swoje `Student ID`.
1. Kliknij „Request permissions” i przejdź cały przepływ.
1. Gdy ID + wymagane uprawnienia są OK, aplikacja sama wyśle odpowiedź dla `G01`.


# G02 — APK / bundle provenance check

## Teoria
Każda aplikacja Android ma co najmniej dwa istotne identyfikatory:
- `applicationId` / `packageName`, czyli nazwę pakietu,
- tożsamość podpisu, czyli certyfikat użyty do podpisania APK lub AAB.

Dla bezpieczeństwa drugi z tych elementów jest ważniejszy. Dwie aplikacje mogą mieć podobną nazwę, podobny interfejs, a nawet zbliżony kod, ale jeżeli nie są podpisane oczekiwanym kluczem, to z punktu widzenia zaufania nie są „tą samą aplikacją”.

W praktyce atak repackagingu wygląda tak:
1. atakujący bierze oryginalne APK,
2. modyfikuje kod albo zasoby,
3. podpisuje zmieniony build własnym kluczem,
4. rozpowszechnia zmodyfikowaną wersję jako pozornie tę samą aplikację.

Z tego powodu w aplikacjach mobilnych rozróżnia się dwa etapy zaufania:
- **install-time trust**: Android sprawdza podpis przy instalacji i aktualizacji aplikacji,
- **runtime trust**: sama aplikacja albo backend wykonują dodatkową kontrolę i decydują, czy temu konkretnemu buildowi wolno zaufać.

To zadanie dotyczy właśnie drugiego etapu. Masz zbudować prosty mechanizm provenance check, który:
- pobiera informację o podpisie aktualnie uruchomionej aplikacji,
- zamienia ją na stabilny identyfikator, np. fingerprint SHA-256,
- porównuje ją z oczekiwaną wartością referencyjną,
- odrzuca build, który nie pasuje,
- odróżnia „Android pozwolił aplikacji się uruchomić” od „moja logika runtime uznała ten build za zaufany”.

## Co masz zbudować
Masz dodać do startera przepływ weryfikacji pochodzenia builda. Końcowy efekt ma być taki, że aplikacja potrafi wyznaczyć stan `ProvenanceState` i na tej podstawie podjąć decyzję bezpieczeństwa.

Stan ma opisywać trzy rzeczy:
- czy tożsamość podpisu zgadza się z oczekiwaną wartością,
- czy build wygląda na zmodyfikowany / niezgodny z oczekiwanym wydawcą,
- czy logika runtime jest wyraźnie oddzielona od samego faktu, że system pozwolił aplikacji się zainstalować i uruchomić.

Nie chodzi o napisanie „magicznego if-a”, tylko o rzeczywiste pobranie danych z systemu i wyciągnięcie z nich wniosku.

## Co otworzyć w projekcie
1. `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`
Tu jest model `ProvenanceState` i miejsce, w którym finalnie ma być podjęta decyzja logiczna.

2. `app/src/main/java/com/example/secretlab/MainActivity.kt`
Ten plik jest dobrym miejscem, żeby osadzić pomocniczy kod runtime albo wywołać helper odpowiedzialny za sprawdzenie podpisu. Zwróć uwagę, że starter ma już importy `PackageManager` i `SigningInfo`, więc kierunek zadania jest już zasugerowany przez kod.

## Jak to zrobić krok po kroku
1. Pobierz `PackageManager` dla aktualnego kontekstu aplikacji.
2. Odczytaj informacje o własnym pakiecie przez `getPackageInfo(...)` z flagą `GET_SIGNING_CERTIFICATES`.
3. Przejdź do `PackageInfo.signingInfo`.
4. Pobierz podpisy z `SigningInfo.getApkContentsSigners()`.
5. Dla podpisu, który uznajesz za aktywną tożsamość builda, pobierz surowe bajty przez `Signature.toByteArray()`.
6. Policz fingerprint SHA-256 z tych bajtów.
7. Przygotuj oczekiwaną wartość referencyjną fingerprintu. To ma być wartość, z którą porównujesz uruchomiony build, a nie wartość obliczona „z tego samego miejsca” chwilę wcześniej tylko po to, żeby test przeszedł.
8. Jeżeli fingerprint pasuje, ustaw `signingIdentityMatchesExpected = true`.
9. Jeżeli fingerprint nie pasuje albo stan danych o podpisie jest niespójny, ustaw `buildLooksTampered = true`.
10. Dopiero po wykonaniu tego sprawdzenia ustaw `installTimeTrustIsSeparatedFromRuntimeTrust = true`. To pole ma znaczyć: „moja logika runtime wykonała własną kontrolę”, a nie „Android kiedyś coś sprawdził przy instalacji”.
11. Na końcu zbuduj `ProvenanceState` i użyj go w `task2Check(...)`.

## Czego nie robić
- Nie sprawdzaj tylko `packageName`.
- Nie ustawiaj wyniku na stałe bez odczytu podpisu z systemu.
- Nie traktuj samego uruchomienia aplikacji jako dowodu, że build jest zaufany.
- Nie mieszaj „czy podpis jest poprawny” z „czy test oczekuje true”. Najpierw ma być logika bezpieczeństwa, dopiero potem wynik testu.

## Weryfikacja
Po implementacji uruchom:
1. `lesson_g_app` -> `app` -> `Tasks` -> `verification` -> `testDebugUnitTest` w Android Studio,
2. albo w terminalu, w katalogu `student/apps/lesson_g_app`:
   `./gradlew :app:bsmEvidence`

Jeżeli zadanie jest wykonane poprawnie, zobaczysz 5-znakowy kod dla `G02`.

## Dokumentacja
- `PackageManager`: https://developer.android.com/reference/android/content/pm/PackageManager
- `PackageManager.getPackageInfo(...)`: https://developer.android.com/reference/android/content/pm/PackageManager#getPackageInfo(java.lang.String,int)
- `PackageInfo.signingInfo`: https://developer.android.com/reference/android/content/pm/PackageInfo#signingInfo
- `SigningInfo`: https://developer.android.com/reference/android/content/pm/SigningInfo
- `SigningInfo.getApkContentsSigners()`: https://developer.android.com/reference/android/content/pm/SigningInfo#getApkContentsSigners()
- `Signature.toByteArray()`: https://developer.android.com/reference/android/content/pm/Signature#toByteArray()
- `MessageDigest`: https://developer.android.com/reference/java/security/MessageDigest

## Format odpowiedzi
Do `final_answer` wpisujesz tylko 5-znakowy kod wypisany przez `:app:bsmEvidence`.


In [ ]:
#@title G02 — Formularz odpowiedzi
code_g02 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g02.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G02", final_answer)


# G03 — Integrity-gated backend request

## Teoria
Sprawdzenie pochodzenia aplikacji nie wystarcza jeszcze do tego, żeby backend zaufał każdemu żądaniu wysłanemu przez klienta. W prawdziwych systemach backend zwykle oczekuje dwóch rzeczy jednocześnie:
- sygnału, że klient działa w zaufanym stanie,
- powiązania żądania z tożsamością aplikacji, tak aby nie dało się łatwo odtworzyć tego samego requestu z innego, niezaufanego klienta.

W tym zadaniu modelujesz właśnie taki przepływ. Nie implementujesz pełnego komercyjnego systemu typu Play Integrity, ale budujesz jego uproszczoną, testowalną wersję.

W modelu startera request ma zostać dopuszczony tylko wtedy, gdy jednocześnie:
- `verdict` ma wartość pozwalającą na wykonanie operacji,
- tożsamość aplikacji zgadza się z tym, czego oczekuje logika bezpieczeństwa,
- request jest związany z tą tożsamością, a nie tylko wysłany na poprawny endpoint z poprawnym `taskId`.

To rozróżnienie jest ważne. Możesz mieć:
- poprawny build, ale request bez bindingu,
- poprawny binding, ale brak zaufanego verdictu,
- poprawny payload biznesowy, ale klient, któremu nie należy ufać.

Dopiero suma tych warunków powinna dawać zgodę na wysyłkę.

## Co masz zbudować
Masz dodać do startera bramkę bezpieczeństwa przed wykonaniem requestu backendowego. Ta bramka ma wyznaczać `IntegrityState`, a następnie blokować albo dopuszczać request.

Końcowy efekt ma być taki, że aplikacja:
- oblicza stan integralności / zaufania,
- podejmuje decyzję jeszcze przed wysłaniem danych,
- w razie braku zaufania nie wysyła nic i pokazuje bezpieczny fallback,
- nie traktuje sieciowej wysyłki jako czegoś, co „i tak warto spróbować”.

## Co otworzyć w projekcie
1. `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`
Tu masz model `IntegrityState` i miejsce, w którym finalnie ma zostać ocenione, czy stan jest poprawny.

2. `app/src/main/java/com/example/secretlab/MainActivity.kt`
Znajdź funkcje `submitTask1(...)` i `submitAnswer(...)`. To jest istotne rozróżnienie:
- `submitTask1(...)` obsługuje auto-submit dla `G01`,
- `submitAnswer(...)` jest ogólną ścieżką HTTP, którą możesz potraktować jako miejsce docelowe dla dodatkowej bramki bezpieczeństwa.

## Jak to zrobić krok po kroku
1. Zdefiniuj, skąd w Twojej implementacji bierze się `verdict`.
   Nie musi to być zewnętrzna usługa. W tym labie ma to być spójny, testowalny sygnał logiczny, który reprezentuje decyzję „pozwól / nie pozwól”.
2. Zdefiniuj, jak potwierdzasz `appPackageNameMatches`.
   To pole samo w sobie nie wystarcza do zaufania, ale jest jednym z warunków kontroli.
3. Dodaj binding requestu do tożsamości aplikacji.
   Oznacza to, że request nie może być akceptowany wyłącznie na podstawie `studentId`, `taskId` i treści odpowiedzi. Musi zawierać albo wyliczać dodatkową informację związaną z aktualną, zaufaną tożsamością builda.
4. Zastanów się, gdzie ten binding wyliczyć.
   W aktualnym starterze najwygodniej zrobić to przed `submitAnswer(...)`, bo w tym miejscu wciąż możesz zdecydować, że request nie powinien zostać wysłany.
5. Zbuduj `IntegrityState` z trzech warunków:
- jaki jest verdict,
- czy nazwa pakietu / tożsamość klienta zgadza się z oczekiwaniami,
- czy request rzeczywiście jest związany z tym klientem.
6. Wywołaj `task3Check(...)` zanim dojdzie do połączenia HTTP.
7. Jeżeli wynik jest negatywny:
- nie wywołuj `submitAnswer(...)`,
- ustaw czytelny komunikat błędu albo brak zaufania w UI,
- nie stosuj ukrytej ścieżki zapasowej,
- nie ustawiaj lokalnego statusu sukcesu bez realnej wysyłki.
8. Jeżeli wynik jest pozytywny, dopiero wtedy dopuść wykonanie requestu.

## Jak myśleć o tym zadaniu
- G02 odpowiada na pytanie: „czy uruchomiony build wygląda na właściwy?”.
- G03 odpowiada na pytanie: „czy temu konkretnemu requestowi można zaufać?”.
- To nie są te same decyzje.
- Poprawny `verdict` bez bindingu requestu nie wystarcza.
- Poprawny binding bez logicznego verdictu też nie wystarcza.

## Czego nie robić
- Nie sprowadzaj zadania do jednego `if (verdict == "ALLOW")`.
- Nie zostawiaj sytuacji, w której request idzie dalej mimo niespełnionego bindingu.
- Nie mieszaj G03 z G01. Auto-submit dla Task 1 ma zostać osobną ścieżką.
- Nie buduj „backend check” jako czysto kosmetycznej flagi, która niczego nie blokuje.

## Weryfikacja
Po implementacji uruchom:
1. `lesson_g_app` -> `app` -> `Tasks` -> `verification` -> `testDebugUnitTest` w Android Studio,
2. albo w terminalu, w katalogu `student/apps/lesson_g_app`:
   `./gradlew :app:bsmEvidence`

Jeżeli zadanie jest wykonane poprawnie, zobaczysz 5-znakowy kod dla `G03`.

## Dokumentacja
- `HttpURLConnection`: https://developer.android.com/reference/java/net/HttpURLConnection
- `URL`: https://developer.android.com/reference/java/net/URL
- `URL.openConnection()`: https://developer.android.com/reference/java/net/URL#openConnection()

## Format odpowiedzi
Do `final_answer` wpisujesz tylko 5-znakowy kod wypisany przez `:app:bsmEvidence`.


In [ ]:
#@title G03 — Formularz odpowiedzi
code_g03 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g03.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G03", final_answer)


# G04 — Backup, migracja i higiena sekretów

## Teoria
W aplikacjach mobilnych sekrety mają cykl życia. Nie wystarczy wiedzieć, jak je odczytać w czasie działania programu. Trzeba jeszcze wiedzieć:
- skąd trafiają do aplikacji,
- gdzie są przechowywane,
- czy są zapisywane lokalnie,
- czy mogą zostać przeniesione na inne urządzenie,
- czy są przypadkiem ujawniane przez backup, migrację albo logowanie diagnostyczne.

To zadanie dotyczy właśnie tej warstwy. Masz przeanalizować i uporządkować politykę obsługi sekretów w starterze tak, żeby dane wrażliwe nie były przypadkowo traktowane jak zwykłe dane aplikacji.

Aktualny starter ma dwa różne modele pracy z sekretami:
- model **build-time**: sekret trafia do aplikacji przez `local.properties` -> `build.gradle.kts` -> `BuildConfig` -> `AppSecrets` -> natywny helper w `secret_keys.cpp`,
- model **runtime/local storage**: sekret jest przechowywany lokalnie przez `AppSecretsStore` i `SecurePrefs`.

To rozróżnienie jest tutaj kluczowe. Build-time secret i runtime secret nie mają tych samych własności bezpieczeństwa. Build-time secret jest częścią procesu budowania aplikacji. Runtime secret może zostać zapisany na urządzeniu i wejść w interakcję z backupem, migracją, reinstalacją albo debugowaniem.

Dodatkowo trzeba pamiętać o jeszcze jednej rzeczy: to, że jakaś wartość jest ukryta za `BuildConfig`, zakodowana w blobie albo odszyfrowywana przez kod natywny, nie rozwiązuje automatycznie problemu backupu. Backup dotyczy przede wszystkim danych zapisanych przez aplikację na urządzeniu.

## Co masz zbudować
Masz uporządkować politykę sekretów i backupu w starterze.

Końcowy efekt ma być taki, że:
- rozumiesz, które sekrety są dostarczane do builda, a które powstają albo są przechowywane lokalnie,
- potrafisz wskazać, które dane nie powinny migrować między urządzeniami,
- poprawiasz konfigurację backupu tak, aby dane wrażliwe nie były objęte pustą, domyślną polityką,
- nie traktujesz „ukrycia sekretu w natywnym helperze” jako zastępstwa dla poprawnej polityki przechowywania i migracji.

## Co otworzyć w projekcie
1. `student/apps/lesson_g_app/local.properties`
Tu znajdują się wartości wejściowe przekazywane do buildu. Zwróć uwagę na `task4_secret_b64` oraz brak ustawionego `map_api_key_b64`.

2. `app/build.gradle.kts`
Znajdź `buildConfigField(...)` dla `MAP_API_KEY_B64` i `TASK4_SECRET_B64`. To jest miejsce, w którym sekret opuszcza warstwę pliku lokalnego i trafia do konfiguracji aplikacji.

3. `app/src/main/java/com/example/secretlab/secure/AppSecrets.kt`
Tu zobaczysz, że sekrety nie są już tylko dekodowane z Base64. Aktualna wersja startera woła `decryptBlob(...)`.

4. `app/src/main/cpp/secret_keys.cpp`
To tutaj jest rzeczywista logika odszyfrowania bloba. Przejdź funkcję `Java_com_example_secretlab_secure_AppSecrets_decryptBlob(...)` krok po kroku:
- wejściowy string jest najpierw dekodowany z Base64,
- wynik trafia do `xxteaDecrypt(...)`,
- dopiero końcowy plaintext wraca do warstwy Kotlin.

5. `app/src/main/java/com/example/secretlab/secure/AppSecretsStore.kt`
To jest drugi model: sekrety zapisane lokalnie po stronie aplikacji.

6. `app/src/main/AndroidManifest.xml`
Sprawdź `android:allowBackup="true"`.

7. `app/src/main/res/xml/backup_rules.xml`
Zobacz, że obecne reguły backupu są puste.

8. `app/src/main/java/com/example/secretlab/MainActivity.kt`
Znajdź `ApiMapCard(...)` i `buildStaticMapUrl(...)`. Zobaczysz tam, że aplikacja buduje URL do statycznej mapy Mapbox z parametrem `access_token`, więc potrzebny jest token Mapbox.

## Jak to zrobić krok po kroku
1. Rozdziel analizę na dwa kanały:
- build-time secrets,
- runtime/local storage secrets.
2. Wskaż, które wartości w starterze należą do kanału build-time.
   W aktualnym projekcie są to `MAP_API_KEY_B64` i `TASK4_SECRET_B64`, przechodzące przez `BuildConfig`, `AppSecrets` i natywny helper.
3. Wskaż, które wartości należą do kanału runtime.
   W aktualnym projekcie odpowiada za to `AppSecretsStore` i warstwa `SecurePrefs`.
4. Określ, które dane mogą być bezpiecznie migrowane, a które nie powinny.
   Dla tego ćwiczenia klucz API mapy oraz sekret zadania 4 traktuj jako dane wrażliwe.
5. Ustal, skąd wziąć klucz API mapy.
   Ponieważ `buildStaticMapUrl(...)` używa endpointu Mapbox i parametru `access_token`, potrzebujesz tokenu Mapbox.
6. Dodaj własny token Mapbox do `local.properties` jako `map_api_key_b64`.
   To jest tylko format wejściowy dla startera. Nie traktuj samej nazwy `_b64` jako ochrony. W tej wersji projektu blob jest potem odszyfrowywany natywnie.
7. Oceń, czy obecna konfiguracja backupu jest akceptowalna.
   Przy `android:allowBackup="true"` i pustym `backup_rules.xml` odpowiedź powinna być krytyczna, bo aplikacja nie rozróżnia jeszcze, które dane lokalne wolno kopiować, a których nie.
8. Popraw politykę backupu.
   Masz dwie sensowne drogi:
- wyłączyć backup całkowicie,
- zostawić backup, ale jawnie wykluczyć lokalne dane wrażliwe.
9. Upewnij się, że nie mieszasz build-time secret z runtime secret.
   To, że blob jest ukryty w kodzie natywnym, nie rozwiązuje problemu danych zapisanych lokalnie po stronie aplikacji.
10. Przejrzyj też miejsca, w których sekret mógłby wyciec przez diagnostykę, logi albo debug output. Polityka higieny sekretów nie kończy się na samym backupie.

## Dodatkowa lektura pomocnicza
- https://al-e-shevelev.medium.com/a-secure-way-to-store-api-keys-in-android-applications-238135709067
- https://docs.mapbox.com/help/glossary/access-token/

Te materiały traktuj pomocniczo. Nie chodzi o mechaniczne powtórzenie wzorca „wrzuć sekret do C++ i problem znika”. W tym zadaniu chodzi o całą politykę: skąd sekret trafia do builda, co dzieje się z nim w runtime i czy może zostać przeniesiony na inne urządzenie.

## Weryfikacja
Po wdrożeniu swojej polityki sekretów i backupu uruchom aplikację oraz przejdź ścieżkę sekretu Task 4:
`local.properties` -> `build.gradle.kts` -> `BuildConfig.TASK4_SECRET_B64` -> `AppSecrets.readTask4Secret()` -> `decryptBlob(...)` w `secret_keys.cpp`.

Na końcu odczytaj odszyfrowaną, 5-znakową wartość sekretu Task 4.

## Dokumentacja
- `android:allowBackup`: https://developer.android.com/guide/topics/manifest/application-element#allowbackup
- `android:fullBackupContent`: https://developer.android.com/guide/topics/manifest/application-element#fullBackupContent
- `BuildConfig`: https://developer.android.com/build/gradle-tips#share-custom-fields-and-resource-values-with-your-apps-code
- `System.loadLibrary(...)`: https://developer.android.com/reference/java/lang/System#loadLibrary(java.lang.String)

## Format odpowiedzi
Do `final_answer` wpisujesz tylko 5-znakową wartość sekretu dla `G04`.


In [ ]:
#@title G04 — Formularz odpowiedzi
secret_g04 = ""  #@param {type:"string"}

final_answer = prepare_answer(secret_g04.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G04", final_answer)
